<div style="padding: 20px; border: 2px solid #4CAF50; border-radius: 10px; background-color: #f9f9f9; text-align: center; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;">
  <h1 style="color: #2E7D32; margin-bottom: 10px;">Tugas Temu Kembali Informasi (TKI)</h1>
  <hr style="border: 1px solid #4CAF50; width: 50%;">
  <table style="margin: 0 auto; text-align: left; font-size: 1.1em; border-spacing: 15px 5px;">
    <tr>
      <td><strong>Andri Darmawan</strong></td>
      <td>NIM : 301210004</td>
    </tr>
    <tr>
      <td><strong>Muhammad Fakhrudin</strong></td>
      <td>NIM : 3012310043</td>
    </tr>
  </table>
</div>

***1: Install & Import Data***

In [46]:
#!pip install Sastrawi

import re
import math
import pandas as pd
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine_similarity
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

***2. Load Dataset***

In [47]:
# Load Data dari GitHub
url_data = 'https://github.com/muhfakhrudin/Tugas_TKI/raw/refs/heads/main/data_ketahanan_pangan_clean.xlsx'
df = pd.read_excel(url_data)

# Memastikan kolom 1 adalah Komentar dan kolom 2 adalah Sumber
df.columns = ['Komentar', 'Sumber']

# Bersihkan baris yang tidak memiliki teks komentar
df = df.dropna(subset=['Komentar']).reset_index(drop=True)

print(f"Jumlah total dokumen: {len(df)}")
display(df.head(50))

Jumlah total dokumen: 50


,Komentar,Sumber
0,Kita dukung upaya pemerintah dalam menyejahter...,https://www.instagram.com/p/DXG3ZY6EdUw/
1,Semoga manajemen pangan kita makin tahan banti...,https://www.instagram.com/p/DXG3ZY6EdUw/
2,Stok beras yang stabil sangat membantu buat me...,https://www.instagram.com/p/DXG3ZY6EdUw/
3,Kemenko Pangan membuktikan kinerjanya dalam me...,https://www.instagram.com/p/DXG3ZY6EdUw/
4,"Pantau harga di tingkat pengecer juga ya, biar...",https://www.instagram.com/p/DXG3ZY6EdUw/
5,Terima kasih sudah memastikan dapur rakyat tet...,https://www.instagram.com/p/DXG3ZY6EdUw/
6,Inilah bentuk nyata perlindungan pemerintah te...,https://www.instagram.com/p/DXG3ZY6EdUw/
7,Langkah nyata Kemenko Pangan bikin kita optimi...,https://www.instagram.com/p/DXG3ZY6EdUw/
8,"Kemenko Pangan mantap, stok beras aman sentosa...",https://www.instagram.com/p/DXG3ZY6EdUw/
9,"Stok beras aman, Indonesia pun makin siap mela...",https://www.instagram.com/p/DXG3ZY6EdUw/


***3. PREPROCESSING (CLEANING, TOKENING, STEMMING)***

In [48]:
stemmer = StemmerFactory().create_stemmer()
stopwords = set(StopWordRemoverFactory().get_stop_words())

def cleaning(teks):
    teks = str(teks).lower()
    teks = re.sub(r'\\d+', ' ', teks)
    teks = re.sub(r'[^a-z\\s]', ' ', teks)
    teks = re.sub(r'\\s+', ' ', teks).strip()
    return teks

def tokenisasi(teks):
    return [t for t in str(teks).split() if len(t) > 1]

def hapus_stopword(tokens):
    return [k for k in tokens if k not in stopwords]

def stemming(tokens):
    return [stemmer.stem(k) for k in tokens]

print("Memulai preprocessing kolom Komentar...")

df = df.reset_index(drop=True)
raw_texts = df['Komentar'].astype(str).values

df['hasil_cleaning'] = [cleaning(t) for t in raw_texts]
df['hasil_token'] = [tokenisasi(t) for t in df['hasil_cleaning']]
df['hasil_stopword'] = [hapus_stopword(t) for t in df['hasil_token']]
df['hasil_stemming'] = [stemming(t) for t in df['hasil_stopword']]
df['teks_bersih'] = df['hasil_stemming'].apply(lambda x: ' '.join(x) if x else 'kosong')

print("Preprocessing selesai!\n")

# Menampilkan seluruh 50 data (tanpa .head())
for i, row in df.iterrows():
    print(f"Komentar {i+1}:")
    print(f"Teks Mentah  : {row['Komentar']}")
    print(f"Hasil Clean  : {row['hasil_cleaning']}")
    print(f"Hasil Token  : {row['hasil_token']}")
    print(f"Stop-word    : {row['hasil_stopword']}")
    print(f"Hasil Stem   : {row['hasil_stemming']}")
    print(f"Hasil Akhir  : '{row['teks_bersih']}'")
    print("\n" + "="*80 + "\n")

Memulai preprocessing kolom Komentar...
Preprocessing selesai!

Komentar 1:
Teks Mentah  : Kita dukung upaya pemerintah dalam menyejahterakan rakyat lewat pangan.
Hasil Clean  : kita dukung upaya pemerintah dalam menyejahterakan rakyat lewat pangan
Hasil Token  : ['kita', 'dukung', 'upaya', 'pemerintah', 'dalam', 'menyejahterakan', 'rakyat', 'lewat', 'pangan']
Stop-word    : ['dukung', 'upaya', 'pemerintah', 'menyejahterakan', 'rakyat', 'lewat', 'pangan']
Hasil Stem   : ['dukung', 'upaya', 'perintah', 'sejahtera', 'rakyat', 'lewat', 'pangan']
Hasil Akhir  : 'dukung upaya perintah sejahtera rakyat lewat pangan'


Komentar 2:
Teks Mentah  : Semoga manajemen pangan kita makin tahan banting hadapi gejolak ekonomi.
Hasil Clean  : semoga manajemen pangan kita makin tahan banting hadapi gejolak ekonomi
Hasil Token  : ['semoga', 'manajemen', 'pangan', 'kita', 'makin', 'tahan', 'banting', 'hadapi', 'gejolak', 'ekonomi']
Stop-word    : ['semoga', 'manajemen', 'pangan', 'makin', 'tahan', 'banting

In [49]:
# Hitung total token awal dari hasil cleaning (sebelum stopword & stemming)
total_token_awal = df['hasil_token'].apply(len).sum()
# Hitung total token akhir (setelah preprocessing lengkap)
total_token_akhir = df['hasil_stemming'].apply(len).sum()

# Hitung kosakata unik awal
kosakata_awal = set()
for tokens in df['hasil_token']:
    kosakata_awal.update(tokens)

# Hitung kosakata unik akhir
kosakata_akhir = set()
for tokens in df['hasil_stemming']:
    kosakata_akhir.update(tokens)

# Hitung persentase
reduksi_token = ((total_token_awal - total_token_akhir) / total_token_awal) * 100
reduksi_kosakata = ((len(kosakata_awal) - len(kosakata_akhir)) / len(kosakata_awal)) * 100

print("Analisis Statistik Reduksi Data:")
print("-" * 40)
print(f"Total token awal             : {total_token_awal} kata")
print(f"Total token akhir            : {total_token_akhir} kata")
print(f"Persentase reduksi token     : {reduksi_token:.2f}%")
print(f"Ukuran kosakata unik awal    : {len(kosakata_awal)} kata")
print(f"Ukuran kosakata unik akhir   : {len(kosakata_akhir)} kata")
print(f"Persentase reduksi kosakata  : {reduksi_kosakata:.2f}%")

Analisis Statistik Reduksi Data:
----------------------------------------
Total token awal             : 488 kata
Total token akhir            : 406 kata
Persentase reduksi token     : 16.80%
Ukuran kosakata unik awal    : 247 kata
Ukuran kosakata unik akhir   : 196 kata
Persentase reduksi kosakata  : 20.65%


In [50]:
import numpy as np

# Memastikan variabel tfidf_model tersedia sebelum mengambil fitur
try:
    vocabulary = tfidf_model.get_feature_names_out()

    print(f"Total Kosakata Unik: {len(vocabulary)} kata\n")
    print("Daftar Kata (A-Z):")
    print("-" * 30)

    # Menampilkan kata-kata dalam format kolom agar mudah dibaca
    for i in range(0, len(vocabulary), 8):
        print(", ".join(vocabulary[i:i+8]))
except NameError:
    print("Error: 'tfidf_model' belum didefinisikan. Silakan jalankan sel pada bagian 'MODELING TF-IDF' terlebih dahulu.")

Total Kosakata Unik: 196 kata

Daftar Kata (A-Z):
------------------------------
abdi, akses, akurat, aman, angka, apresiasi, atas, awal
awas, baik, banget, bangga, bangsa, bangun, banting, bantu
banyak, bareng, baru, batas, beli, bener, bentuk, beras
berita, biar, bijak, bikin, buah, buat, bukti, butuh
buying, cadang, canggih, cita, cukup, damba, dampak, damping
dapet, dapur, dasar, data, daulat, daya, dengernya, depan
distribusi, dukung, ecer, ekonomi, el, emang, energi, fokus
fondasi, gabah, gairah, garda, gejolak, gercep, giat, gudang
hadap, hak, harga, hari, hasil, ikut, indonesia, inflasi
ini, inisiatif, inovasi, iring, isu, jadi, jaga, jangan
jangkau, kalau, kasih, kecil, keluarga, kemenko, kendali, keras
kerja, komoditas, kondusif, konsumen, kontrol, kosong, kualitas, kuat
labuh, lain, laju, lama, lambung, lancar, langkah, layak
lewat, lihat, limpah, lindung, lokal, macet, maju, makin
makmur, mana, manajemen, mandiri, mantap, manusia, martabat, masa
masyarakat, mati, mikro, mog

***3. PEMBUKTIAN HITUNGAN MANUAL TF & IDF***

In [ ]:
nomor_urut_sampel = 0
raw_text = df['Komentar'].iloc[nomor_urut_sampel]
hasil_prep = df['hasil_stemming'].iloc[nomor_urut_sampel]
total_kata = len(hasil_prep)
hitung_kata = Counter(hasil_prep)
N = len(df)

# 1. Hitung DF untuk seluruh kata
df_count = {}
for token_list in df['hasil_stemming']:
    for kata in set(token_list):
        df_count[kata] = df_count.get(kata, 0) + 1

print(f"PEMBUKTIAN HITUNGAN MANUAL - Komentar #{nomor_urut_sampel+1}")
print(f"Dokumen Asli : '{raw_text}'")
print(f"Hasil Prep   : {hasil_prep}")
print(f"Total Kata   : {total_kata} | Total Dokumen (N): {N}")

print("\nLangkah 3.1: Tahap Penghitungan Term Frequency (TF)")
print("-" * 70)
print(f"  {'Kata':<20} | {'Jumlah':<8} | {'Rumus (n/Total)':<20} | {'Hasil TF'}")
print("-" * 70)
for kata, jumlah in hitung_kata.items():
    print(f"  {kata:<20} | {jumlah:<8} | {f'{jumlah}/{total_kata}':<20} | {jumlah/total_kata:.4f}")

print("\nLangkah 3.2: Tahap Penghitungan Inverse Document Frequency (IDF)")
print("-" * 70)
print(f"  {'Kata':<20} | {'DF':<8} | {'Rumus (log10 N/df)':<20} | {'Hasil IDF'}")
print("-" * 70)
for kata in hitung_kata.keys():
    idf_val = math.log10(N / df_count[kata])
    print(f"  {kata:<20} | {df_count[kata]:<8} | {f'log10({N}/{df_count[kata]})':<20} | {idf_val:.4f}")

print("\nLangkah 3.3: Pembobotan TF-IDF Manual")
print("-" * 95)
print(f"  {'Kata':<20} | {'TF':<10} | {'IDF':<10} | {'Perhitungan (TF x IDF)':<25} | {'Hasil TF-IDF'}")
print("-" * 95)
for kata, jumlah in hitung_kata.items():
    tf_val = jumlah / total_kata
    idf_val = math.log10(N / df_count[kata])
    tfidf_val = tf_val * idf_val
    print(f"  {kata:<20} | {tf_val:<10.4f} | {idf_val:<10.4f} | {f'{tf_val:.4f} x {idf_val:.4f}':<25} | {tfidf_val:.4f}")

PEMBUKTIAN HITUNGAN MANUAL - Komentar #1
Dokumen Asli : 'Kita dukung upaya pemerintah dalam menyejahterakan rakyat lewat pangan.'
Hasil Prep   : ['dukung', 'upaya', 'perintah', 'sejahtera', 'rakyat', 'lewat', 'pangan']
Total Kata   : 7 | Total Dokumen (N): 50

Langkah 3.1: Tahap Penghitungan Term Frequency (TF)
----------------------------------------------------------------------
  Kata                 | Jumlah   | Rumus (n/Total)      | Hasil TF
----------------------------------------------------------------------
  dukung               | 1        | 1/7                  | 0.1429
  upaya                | 1        | 1/7                  | 0.1429
  perintah             | 1        | 1/7                  | 0.1429
  sejahtera            | 1        | 1/7                  | 0.1429
  rakyat               | 1        | 1/7                  | 0.1429
  lewat                | 1        | 1/7                  | 0.1429
  pangan               | 1        | 1/7                  | 0.1429

Langkah 3.2: T

***4. MODELING TF-IDF & VSM SCIKIT-LEARN***

In [52]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Inisialisasi model
global tfidf_model
tfidf_model = TfidfVectorizer(
    sublinear_tf=True,
    use_idf=True,
    smooth_idf=True,
    norm='l2'
)

# Proses Fit & Transform
matriks_tfidf = tfidf_model.fit_transform(df['teks_bersih'])

print(f"Model TF-IDF Berhasil Dibuat!")
print(f"Dimensi Matriks: {matriks_tfidf.shape[0]} dokumen x {matriks_tfidf.shape[1]} kata unik")

def cari_dokumen(query, top_k=5):
    query_clean = cleaning(query)
    query_tokens = tokenisasi(query_clean)
    query_stop = hapus_stopword(query_tokens)
    query_stem = stemming(query_stop)
    query_final = ' '.join(query_stem) if query_stem else 'kosong'

    query_vector = tfidf_model.transform([query_final])
    skor_kemiripan = sklearn_cosine_similarity(query_vector, matriks_tfidf)[0]
    urutan = skor_kemiripan.argsort()[::-1][:top_k]

    print("=" * 70)
    print(f"Query: '{query}'")
    print("=" * 70)

    for rank, idx in enumerate(urutan, 1):
        skor = skor_kemiripan[idx]
        if skor > 0:
            print(f"[{rank}] Skor: {skor:.4f} | Komentar #{idx+1}")
            print(f"Isi: {df['Komentar'].iloc[idx]}\n")

Model TF-IDF Berhasil Dibuat!
Dimensi Matriks: 50 dokumen x 196 kata unik


In [53]:
#Output Matriks TF-IDF (Scikit-Learn)
#Berikut adalah representasi matriks dalam bentuk tabel untuk verifikasi nilai bobot antar dokumen.
import pandas as pd

# Mengambil nama fitur (kata unik)
fitur_names = tfidf_model.get_feature_names_out()

# Membuat DataFrame dari matriks TF-IDF Scikit-Learn
df_tfidf_vsm = pd.DataFrame(matriks_tfidf.toarray(), columns=fitur_names)

# Mengatur display agar semua kolom terlihat (horizontal) namun baris dibatasi
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 800)

print("MATRIKS TF-IDF (SAMPEL 2 KOMENTAR TERATAS)")
print("=" * 50)
# Mengambil 2 baris pertama saja
print(df_tfidf_vsm.head(2))

print("\n" + "-" * 50)
print(f"Informasi Matriks: {df_tfidf_vsm.shape[0]} Dokumen & {df_tfidf_vsm.shape[1]} Fitur/Kata.")

MATRIKS TF-IDF (SAMPEL 2 KOMENTAR TERATAS)
   abdi  akses  akurat  aman  angka  apresiasi  atas  awal  awas  baik  banget  bangga  bangsa  bangun   banting  bantu  banyak  bareng  baru  batas  beli  bener  bentuk  beras  berita  biar  bijak  bikin  buah  buat  bukti  butuh  buying  cadang  canggih  cita  cukup  damba  dampak  damping  dapet  dapur  dasar  data  daulat  daya  dengernya  depan  distribusi    dukung  ecer   ekonomi   el  emang  energi  fokus  fondasi  gabah  gairah  garda   gejolak  gercep  giat  gudang     hadap  hak  harga  hari  hasil  ikut  indonesia  inflasi  ini  inisiatif  inovasi  iring  isu  jadi  jaga  jangan  jangkau  kalau  kasih  kecil  keluarga  kemenko  kendali  keras  kerja  komoditas  kondusif  konsumen  kontrol  kosong  kualitas  kuat  labuh  lain  laju  lama  lambung  lancar  langkah  layak  \
0   0.0    0.0     0.0   0.0    0.0        0.0   0.0   0.0   0.0   0.0     0.0     0.0     0.0     0.0  0.000000    0.0     0.0     0.0   0.0    0.0   0.0    0.0 

***5. TESTING PENCARIAN & EXPORT HASIL***

In [54]:
kueri_test = "kayaknya pangan kita bermasalah sampai harus MBG"
# Jalankan fungsi pencarian (sekarang akan menampilkan teks lengkap)
cari_dokumen(kueri_test, top_k=5)

# Ekspor tabel hasil preprocessing ke Excel
df_output = df[['Komentar', 'teks_bersih']]
df_output.to_excel('hasil_vsm_ketahanan_pangan.xlsx', index=False)

print("\n[INFO] Tabel data bersih telah disimpan di file 'hasil_vsm_ketahanan_pangan.xlsx'")

Query: 'kayaknya pangan kita bermasalah sampai harus MBG'
[1] Skor: 0.3277 | Komentar #48
Isi: Mantap Kemenko Pangan, gercep terus jaga ketahanan pangan kita.

[2] Skor: 0.3187 | Komentar #39
Isi: Sukses terus Kemenko Pangan dalam mengawal kedaulatan pangan nasional.

[3] Skor: 0.1954 | Komentar #16
Isi: Ketahanan pangan yang kuat bikin bangsa kita makin bermartabat.

[4] Skor: 0.1932 | Komentar #33
Isi: Teruslah mengabdi untuk memastikan kedaulatan pangan tetap terjaga.

[5] Skor: 0.1902 | Komentar #19
Isi: Ketegasan pemerintah dalam mengawasi stok pangan sangat diapresiasi.


[INFO] Tabel data bersih telah disimpan di file 'hasil_vsm_ketahanan_pangan.xlsx'


# Nomor 1

### PEMBUATAN INVERTED INDEX

In [55]:
from collections import Counter

# Inisialisasi Inverted Index
inverted_index = {}

# Membangun indeks dari kolom hasil_stemming
for doc_id, tokens in enumerate(df['hasil_stemming']):
    hitung_kata = Counter(tokens)
    for term, tf_mentah in hitung_kata.items():
        if term not in inverted_index:
            inverted_index[term] = {}
        # Simpan doc_id beserta frekuensi mentah kata tersebut
        inverted_index[term][doc_id] = tf_mentah

print(f"Inverted Index berhasil dibuat!")
# Menampilkan sampel 5 kata pertama di dalam Inverted Index
sampel_kata = list(inverted_index.keys())[:5]
print("\nSampel isi Inverted Index (5 kata pertama):")
for kata in sampel_kata:
    print(f"Kata '{kata}' -> Posting List (Doc_ID: TF Menta): {inverted_index[kata]}")

Inverted Index berhasil dibuat!

Sampel isi Inverted Index (5 kata pertama):
Kata 'dukung' -> Posting List (Doc_ID: TF Menta): {0: 1, 17: 1}
Kata 'upaya' -> Posting List (Doc_ID: TF Menta): {0: 1}
Kata 'perintah' -> Posting List (Doc_ID: TF Menta): {0: 1, 6: 1, 16: 1, 18: 1, 37: 1, 43: 1}
Kata 'sejahtera' -> Posting List (Doc_ID: TF Menta): {0: 1, 24: 1}
Kata 'rakyat' -> Posting List (Doc_ID: TF Menta): {0: 1, 5: 1, 26: 1, 37: 1}


### PERHITUNGAN MANUAL LOG FREQUENCY

In [56]:
import math
from collections import Counter

nomor_urut_sampel = 0  # Komentar #1
raw_text = df['Komentar'].iloc[nomor_urut_sampel]
hasil_prep = df['hasil_stemming'].iloc[nomor_urut_sampel]
total_kata = len(hasil_prep)
hitung_kata = Counter(hasil_prep)
N = len(df)

# Hitung Document Frequency (df) dari seluruh koleksi dokumen
df_count = {kata: len(posting) for kata, posting in inverted_index.items()}

print(f"PEMBUKTIAN HITUNGAN MANUAL (LOG FREQUENCY) - Komentar #{nomor_urut_sampel+1}")
print(f"Dokumen Asli : '{raw_text}'")
print(f"Hasil Prep   : {hasil_prep}")
print(f"Total Dokumen (N): {N}\n")

print("Langkah 2.1: Tahap Penghitungan Term Frequency (TF Log)")
print("-" * 75)
print(f"  {'Kata':<20} | {'TF Mentah':<10} | {'Rumus TF (1 + log10(tf))':<25} | {'Hasil TF Log'}")
print("-" * 75)
for kata, tf_mentah in hitung_kata.items():
    tf_log = 1 + math.log10(tf_mentah) if tf_mentah > 0 else 0
    print(f"  {kata:<20} | {tf_mentah:<10} | {f'1 + log10({tf_mentah})':<25} | {tf_log:.4f}")

print("\nLangkah 2.2: Tahap Penghitungan Inverse Document Frequency (IDF)")
print("-" * 75)
print(f"  {'Kata':<20} | {'DF':<10} | {'Rumus IDF (log10(N/df))':<25} | {'Hasil IDF'}")
print("-" * 75)
for kata in hitung_kata.keys():
    idf_val = math.log10(N / df_count[kata])
    print(f"  {kata:<20} | {df_count[kata]:<10} | {f'log10({N}/{df_count[kata]})':<25} | {idf_val:.4f}")

print("\nLangkah 2.3: Pembobotan TF-IDF Manual (Log Frequency Weighting)")
print("-" * 95)
print(f"  {'Kata':<20} | {'TF Log':<10} | {'IDF':<10} | {'Perhitungan (TF x IDF)':<25} | {'Hasil TF-IDF'}")
print("-" * 95)
for kata, tf_mentah in hitung_kata.items():
    tf_log = 1 + math.log10(tf_mentah) if tf_mentah > 0 else 0
    idf_val = math.log10(N / df_count[kata])
    tfidf_val = tf_log * idf_val
    print(f"  {kata:<20} | {tf_log:<10.4f} | {idf_val:<10.4f} | {f'{tf_log:.4f} x {idf_val:.4f}':<25} | {tfidf_val:.4f}")

PEMBUKTIAN HITUNGAN MANUAL (LOG FREQUENCY) - Komentar #1
Dokumen Asli : 'Kita dukung upaya pemerintah dalam menyejahterakan rakyat lewat pangan.'
Hasil Prep   : ['dukung', 'upaya', 'perintah', 'sejahtera', 'rakyat', 'lewat', 'pangan']
Total Dokumen (N): 50

Langkah 2.1: Tahap Penghitungan Term Frequency (TF Log)
---------------------------------------------------------------------------
  Kata                 | TF Mentah  | Rumus TF (1 + log10(tf))  | Hasil TF Log
---------------------------------------------------------------------------
  dukung               | 1          | 1 + log10(1)              | 1.0000
  upaya                | 1          | 1 + log10(1)              | 1.0000
  perintah             | 1          | 1 + log10(1)              | 1.0000
  sejahtera            | 1          | 1 + log10(1)              | 1.0000
  rakyat               | 1          | 1 + log10(1)              | 1.0000
  lewat                | 1          | 1 + log10(1)              | 1.0000
  pangan         

### MESIN PENCARI VSM KUSTOM & NORMALISASI

In [57]:
import numpy as np

# 1. Daftarkan seluruh kosakata unik (fitur) dari indeks
kosakata_vsm = sorted(list(inverted_index.keys()))
dimensi_kata = len(kosakata_vsm)
indeks_kata = {kata: i for i, kata in enumerate(kosakata_vsm)}

# 2. Hitung IDF untuk semua kata global
N_docs = len(df)
global_idf = {}
for kata, posting in inverted_index.items():
    global_idf[kata] = math.log10(N_docs / len(posting))

# 3. Membangun Matriks TF-IDF Dokumen berbasis Log Weighting
matriks_tfidf_kustom = np.zeros((N_docs, dimensi_kata))
for kata, posting in inverted_index.items():
    kata_idx = indeks_kata[kata]
    idf_w = global_idf[kata]
    for doc_id, tf_mentah in posting.items():
        tf_log = 1 + math.log10(tf_mentah) if tf_mentah > 0 else 0
        matriks_tfidf_kustom[doc_id, kata_idx] = tf_log * idf_w

# 4. Fungsi Pencarian Komparatif VSM
def mesin_pencari_vsm(query, top_k=5):
    # Preprocessing kueri
    q_clean = cleaning(query)
    q_tokens = tokenisasi(q_clean)
    q_stop = hapus_stopword(q_tokens)
    q_stem = stemming(q_stop)
    
    # Hitung TF-IDF Vektor Kueri
    q_counts = Counter(q_stem)
    vektor_query = np.zeros(dimensi_kata)
    for kata, tf_mentah in q_counts.items():
        if kata in indeks_kata:
            kata_idx = indeks_kata[kata]
            tf_log = 1 + math.log10(tf_mentah) if tf_mentah > 0 else 0
            vektor_query[kata_idx] = tf_log * global_idf[kata]
            
    # Perhitungan Kemiripan Jarak/Sudut
    # A. Tanpa Normalisasi (Hanya Dot Product)
    skor_tanpa_norm = np.dot(matriks_tfidf_kustom, vektor_query)
    
    # B. Menggunakan Cosine Normalization
    norm_query = np.linalg.norm(vektor_query)
    norm_dokumen = np.linalg.norm(matriks_tfidf_kustom, axis=1)
    
    # Hindari pembagian dengan nol jika ada dokumen kosong setelah preprocessing
    norm_dokumen[norm_dokumen == 0] = 1.0
    if norm_query == 0:
        skor_cosine = np.zeros(N_docs)
    else:
        skor_cosine = skor_tanpa_norm / (norm_query * norm_dokumen)
        
    # Tampilkan Perbandingan Hasil Ranked Retrieval
    urutan_cosine = skor_cosine.argsort()[::-1][:top_k]
    urutan_tanpa_norm = skor_tanpa_norm.argsort()[::-1][:top_k]
    
    print("=" * 85)
    print(f"QUERY PENGGUNA: '{query}'")
    print("=" * 85)
    
    print("\n--- HASIL DENGAN COSINE NORMALIZATION (RANKED RETAIVAL) ---")
    for rank, idx in enumerate(urutan_cosine, 1):
        if skor_cosine[idx] > 0:
            print(f"[{rank}] Skor Cosine: {skor_cosine[idx]:.4f} | Dokumen #{idx+1}: {df['Komentar'].iloc[idx][:70]}...")
            
    print("\n--- HASIL TANPA NORMALISASI (DOT PRODUCT) ---")
    for rank, idx in enumerate(urutan_tanpa_norm, 1):
        if skor_tanpa_norm[idx] > 0:
            print(f"[{rank}] Skor Dot-Prod: {skor_tanpa_norm[idx]:.4f} | Dokumen #{idx+1}: {df['Komentar'].iloc[idx][:70]}...")


### Uji coba fungsi mesin pencari kustom

In [ ]:
# mesin_pencari_vsm("pangan aman harga stabil", top_k=3)

print("     MINI SEARCH ENGINE VSM - KETAHANAN PANGAN           ")
print("\nSistem siap digunakan. Ketik 'keluar' untuk berhenti.\n")

while True:
    # 1. Menerima input kueri secara langsung dari pengguna
    kueri_pengguna = input("Masukkan kueri pencarian Anda: ")
    
    # Kondisi untuk menghentikan program
    if kueri_pengguna.lower() == 'keluar':
        print("Program dihentikan. Terima kasih!")
        break
        
    # Validasi jika pengguna hanya menekan enter (input kosong)
    if not kueri_pengguna.strip():
        print("Kueri tidak boleh kosong! Silakan masukkan kata kunci.\n")
        continue
        
    # 2. Menjalankan pencarian ranked retrieval (Top 5 dokumen tertinggi)
    mesin_pencari_vsm(kueri_pengguna, top_k=5)
    print("\n" + "-" * 85 + "\n")

     MINI SEARCH ENGINE VSM - KETAHANAN PANGAN           
Sistem siap digunakan. Ketik 'keluar' untuk berhenti.

QUERY PENGGUNA: 'harga beras stabil dan murah'

--- HASIL DENGAN COSINE NORMALIZATION (RANKED RETAIVAL) ---
[1] Skor Cosine: 0.3682 | Dokumen #42: Kestabilan harga pangan bikin ekonomi mikro makin bergairah....
[2] Skor Cosine: 0.3254 | Dokumen #5: Pantau harga di tingkat pengecer juga ya, biar stabil sampai ke tangan...
[3] Skor Cosine: 0.3083 | Dokumen #41: Beras stabil harga mati, terima kasih atas kerja kerasnya selama ini....
[4] Skor Cosine: 0.2383 | Dokumen #20: Berita stok beras stabil bikin suasana ekonomi jadi makin kondusif....
[5] Skor Cosine: 0.2304 | Dokumen #3: Stok beras yang stabil sangat membantu buat menekan angka inflasi....

--- HASIL TANPA NORMALISASI (DOT PRODUCT) ---
[1] Skor Dot-Prod: 1.7312 | Dokumen #41: Beras stabil harga mati, terima kasih atas kerja kerasnya selama ini....
[2] Skor Dot-Prod: 1.5546 | Dokumen #42: Kestabilan harga pangan bikin ek

# Nomor 2

### 2a. Analisis Bobot (IDF)

Berdasarkan hasil eksekusi program pada "Langkah 3.2: Tahap Penghitungan Inverse Document Frequency (IDF)" untuk dokumen sampel (Komentar #1), diambil 2 kata kunci dari topik Ketahanan Pangan dengan nilai $df$ (*Document Frequency*) yang berbeda untuk dianalisis:

1. **Kata kunci "pangan"**
   * Jumlah total dokumen ($N$) = $50$
   * Jumlah dokumen yang mengandung kata tersebut ($df$) = $25$
   * **Perhitungan Manual Langkah demi Langkah:**
     $$IDF(\text{"pangan"}) = \log_{10}\left(\frac{N}{df}\right)$$
     $$IDF(\text{"pangan"}) = \log_{10}\left(\frac{50}{25}\right) = \log_{10}(2) \approx 0.3010$$

2. **Kata kunci "upaya"**
   * Jumlah total dokumen ($N$) = $50$
   * Jumlah dokumen yang mengandung kata tersebut ($df$) = $1$
   * **Perhitungan Manual Langkah demi Langkah:**
     $$IDF(\text{"upaya"}) = \log_{10}\left(\frac{N}{df}\right)$$
     $$IDF(\text{"upaya"}) = \log_{10}\left(\frac{50}{1}\right) = \log_{10}(50) \approx 1.6990$$

#### Jelaskan mengapa kata yang lebih jarang muncul memiliki bobot lebih tinggi?

Secara matematis, nilai $df$ berada pada posisi penyebut (pembagi) di dalam rumus IDF:
$$\log_{10}\left(\frac{N}{df}\right)$$

* Jika suatu kata sangat sering muncul (seperti kata **"pangan"** yang muncul di $25$ dari $50$ dokumen), nilai pembagi menjadi besar sehingga rasio $\frac{N}{df}$ mengecil mendekati angka $1$. Nilai $\log_{10}(1)$ adalah $0$. Artinya, kata tersebut dianggap sebagai kata umum (*common word*) yang tidak memiliki kekuatan diskriminatif (tidak bisa membedakan uniknya satu dokumen dengan dokumen lain).
* Sebaliknya, jika suatu kata sangat jarang muncul (seperti kata **"upaya"** yang hanya muncul di $1$ dokumen), nilai pembagi sangat kecil sehingga rasio $\frac{N}{df}$ membengkak menjadi besar ($50$). Logaritma dari angka besar menghasilkan nilai bobot yang tinggi ($1.6990$).

**Kesimpulan:** Kata yang jarang muncul memiliki kadar informasi (*information gain*) yang jauh lebih tinggi dan spesifik, sehingga sistem IR memberikan penghargaan berupa bobot yang lebih tinggi untuk membantu proses penemuan dokumen yang spesifik pula.

### 2b. Analisis Efek Normalisasi dalam Vector Space Model (VSM)


Di dalam sistem *Information Retrieval* berbasis *Vector Space Model* (VSM), kemiripan antara kueri ($Q$) dan dokumen ($D$) dihitung menggunakan metode *Cosine Similarity*. Rumus lengkapnya adalah:

$$\text{Cosine Similarity}(Q, D) = \frac{\vec{Q} \cdot \vec{D}}{\|\vec{Q}\| \times \|\vec{D}\|}$$

Di mana bagian pembilang ($\vec{Q} \cdot \vec{D}$) merupakan operasi *Dot Product* mentah, sedangkan bagian penyebut ($\|\vec{Q}\| \times \|\vec{D}\|$) adalah proses **Cosine Normalization** (Normalisasi Panjang Vektor).

#### Mengapa dokumen yang sangat panjang cenderung memiliki skor lebih tinggi jika tidak dinormalisasi?

1. **Efek Frekuensi Kata (Term Frequency Bias):**
   Dokumen yang sangat panjang secara alamiah memiliki jumlah kata yang melimpah. Hal ini memperbesar peluang terjadinya pengulangan kata kunci kunci kueri di dalam dokumen tersebut, sehingga nilai $tf$ mentahnya tinggi.

2. **Skor Pembilang yang Terus Akumulatif (*Length Bias*):**
   Jika kita **tidak menggunakan normalisasi**, skor kemiripan murni hanya dihitung berdasarkan nilai *Dot Product* saja:
   $$\text{Skor}_{\text{tanpa norm}} = \sum_{t \in Q \cap D} w_{t,Q} \times w_{t,D}$$
   Karena dokumen panjang memiliki bobot $w_{t,D}$ yang besar dan variasi kata yang banyak, hasil penjumlahan (*summation*) dari perkalian vektor tersebut akan terus membengkak. Dokumen panjang akan selalu mendominasi peringkat teratas mesin pencari, mengalahkan dokumen pendek meskipun dokumen pendek tersebut sebenarnya jauh lebih padat dan relevan.

3. **Peran Krusial Cosine Normalization:**
   *Cosine Normalization* bertindak sebagai "penalti" yang adil dengan membagi hasil *dot product* dengan panjang Euclidean dari dokumen tersebut ($\|\vec{D}\|$). 
   
   Panjang vektor dokumen dihitung dari:
   $$\|\vec{D}\| = \sqrt{w_{1}^2 + w_{2}^2 + \dots + w_{n}^2}$$
   
   Dengan membaginya dengan $\|\vec{D}\|$, ukuran dokumen diproyeksikan ke dalam unit panjang yang setara (satuan radius ruang vektor = 1). Hasilnya, dokumen pendek yang fokus membahas kueri akan mendapatkan skor *cosine* yang tinggi, sementara dokumen panjang yang bertele-tele akan diredam skornya karena nilai pembaginya ($\|\vec{D}\|$) yang besar.

### 2C Evaluasi Sistem

In [59]:
from IPython.display import clear_output
import numpy as np
import math
from collections import Counter

def dapatkan_top_retrieved(query, top_k=5):
    """Mengambil ID dokumen hasil Cosine Similarity :"""
    q_clean = cleaning(query)
    q_tokens = tokenisasi(q_clean)
    q_stop = hapus_stopword(q_tokens)
    q_stem = stemming(q_stop)
    
    q_counts = Counter(q_stem)
    vektor_query = np.zeros(dimensi_kata)
    for kata, tf_mentah in q_counts.items():
        if kata in indeks_kata:
            vektor_query[indeks_kata[kata]] = (1 + math.log10(tf_mentah)) * global_idf[kata]
            
    skor_tanpa_norm = np.dot(matriks_tfidf_kustom, vektor_query)
    norm_query = np.linalg.norm(vektor_query)
    norm_dokumen = np.linalg.norm(matriks_tfidf_kustom, axis=1)
    norm_dokumen[norm_dokumen == 0] = 1.0
    
    skor_cosine = skor_tanpa_norm / (norm_query * norm_dokumen) if norm_query > 0 else np.zeros(N_docs)
    urutan_cosine = skor_cosine.argsort()[::-1]
    
    return [idx for idx in urutan_cosine if skor_cosine[idx] > 0][:top_k]

def hitung_metrik(retrieved, ground_truth):
    retrieved_set = set(retrieved)
    gt_set = set(ground_truth)
    relevan_terpanggil = retrieved_set.intersection(gt_set)
    
    precision = len(relevan_terpanggil) / len(retrieved_set) if len(retrieved_set) > 0 else 0
    recall = len(relevan_terpanggil) / len(gt_set) if len(gt_set) > 0 else 0
    f_measure = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    return precision, recall, f_measure

In [ ]:
# Skenario Pengujian
rekap_hasil = []

for i in range(1, 3):
    # Bersihkan log layar sebelumnya agar VS Code tidak melakukan truncation
    clear_output(wait=True)
    
    print("="*80)
    print(f"             MODUL EVALUASI SISTEM INTERAKTIF - SKENARIO PENGUJIAN #{i}            ")
    print("="*80)
    kueri = input(f"Masukkan Kueri {i}: ")
    
    # Ambil hasil pemanggilan sistem
    retrieved = dapatkan_top_retrieved(kueri, top_k=5)
    
    print(f"\n[Sistem IR] Menemukan {len(retrieved)} dokumen teratas.")
    if retrieved:
        print("Daftar dokumen teratas (TEKS LENGKAP):")
        print("-" * 80)
        for rank, idx in enumerate(retrieved, 1):
            teks_lengkap = df['Komentar'].iloc[idx]
            print(f" {rank}. [ID Dokumen: {idx}]")
            print(f"    Isi: \"{teks_lengkap}\"")
            print("-" * 80)
    else:
        print("  (Tidak ada dokumen yang cocok)\n")
        
    print("\nTentukan Ground Truth untuk evaluasi skenario ini.")
    print("Pilih nomor ID dokumen di atas yang menurut Anda benar-benar relevan.")
    print("-> Jika lebih dari satu, pisahkan dengan koma (Contoh: 41, 4)")
    input_gt = input("Masukkan ID Ground Truth: ")
    
    # Parsing input string ke list integer
    ground_truth = []
    if input_gt.strip():
        ground_truth = [int(x.strip()) for x in input_gt.split(",") if x.strip().isdigit()]
        
    # Hitung metrik akurasi
    p, r, f = hitung_metrik(retrieved, ground_truth)
    
    rekap_hasil.append({
        "kueri": kueri,
        "retrieved": ", ".join(map(str, retrieved)) if retrieved else "-",
        "gt": ", ".join(map(str, ground_truth)) if ground_truth else "-",
        "p": p * 100,
        "r": r * 100,
        "f": f * 100
    })

# SETELAH LOOP SELESAI: BERSIHKAN LAYAR TOTAL & TAMPILKAN TABEL UTUH
clear_output(wait=True)
print("\n" + "="*120)
print("                                   TABEL INDIKATOR EVALUASI AKHIR UTS (POIN 2C)                                ")
print("="*120)
print(f" {'No':<3} | {'Kueri':<35} | {'Retrieved IDs':<18} | {'Ground Truth':<15} | {'Precision':<11} | {'Recall':<11} | {'F1-Score'}")
print("-" * 120)
for idx, h in enumerate(rekap_hasil, 1):
    kueri_display = h['kueri'][:32] + "..." if len(h['kueri']) > 35 else h['kueri']
    print(f" {idx:<3} | {kueri_display:<35} | {h['retrieved']:<18} | {h['gt']:<15} | {h['p']:<10.1f}% | {h['r']:<10.1f}% | {h['f']:.1f}%")
print("="*120)


                                   TABEL INDIKATOR EVALUASI AKHIR UTS (POIN 2C)                                
 No  | Kueri                               | Retrieved IDs      | Ground Truth    | Precision   | Recall      | F1-Score
------------------------------------------------------------------------------------------------------------------------
 1   | harga beras murah                   | 26, 21, 22, 41, 40 | 22, 41          | 40.0      % | 100.0     % | 57.1%
 2   | harga beras tidak stabil            | 41, 4, 40, 19, 2   | 4, 19           | 40.0      % | 100.0     % | 57.1%
